# Versao 10 - Visao Geral Dos Dados

Este notebook abre a `versao10` com uma mudanca de perspectiva. A pergunta central deixa de ser apenas "qual arquitetura recorrente e mais forte?" e passa a ser "como representar o `3W` de forma mais fiel ao problema descrito no artigo do dataset?".

## Direcao metodologica inspirada no artigo

O arquivo [2507.01048v1.pdf](/home/tiagoriosrocha/Desktop/lstm-w3/2507.01048v1.pdf) reposiciona de forma importante a estrategia do projeto. A leitura do artigo indica que o `3W Dataset 2.0.0` foi pensado para um cenario mais rico do que a simples classificacao da instancia como um todo.

O texto descreve, em termos praticos, cinco pistas metodologicas centrais:

1. as instancias sao series temporais multivariadas com `27` variaveis no desenho do dataset;
2. existem rotulos por observacao para `class` e `state`, portanto ha supervisao temporal interna;
3. as classes transitorias `101..109` fazem parte da dinamica do problema e ajudam a representar a progressao do evento;
4. instancias reais preservam `missing values`, `frozen variables` e `outliers`, isto e, o ruído operacional nao deve ser tratado apenas como inconveniente;
5. a origem da amostra (`well`, `simulated`, `drawn`) tambem pode carregar informacao estrutural.

A `versao10` nasce exatamente dessa leitura. Em vez de apenas aprofundar a `LSTM`, a proposta agora e tornar o modelo mais coerente com a forma como o dataset foi concebido.

## Hipotese da versao 10

A hipotese agora e mais ambiciosa e mais bem fundamentada:

- se usarmos as `27` variaveis;
- se preservarmos explicitamente `missing values` e `frozen variables`;
- se aproveitarmos os rotulos sequenciais de `class` e `state`;
- se informarmos ao modelo a origem da amostra;

entao a rede recorrente pode reduzir de forma mais consistente a vantagem das baselines tabulares.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == "versao10" else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from versao10.pipeline_v10 import (
    CONTINUOUS_SENSOR_COLUMNS,
    FULL_FEATURE_COLUMNS,
    OBSERVATION_CLASS_CODES,
    OBSERVATION_STATE_CODES,
    SOURCE_TYPE_MAPPING,
    STATE_SENSOR_COLUMNS,
    discover_series_manifest,
    load_attribute_catalog,
    load_event_catalog,
)

import pandas as pd

DATASET_ROOT = PROJECT_ROOT / "3W" / "dataset"
PDF_PATH = PROJECT_ROOT / "2507.01048v1.pdf"

In [2]:
attribute_catalog = load_attribute_catalog(DATASET_ROOT)
event_catalog = load_event_catalog(DATASET_ROOT)
manifest = discover_series_manifest(DATASET_ROOT)

print("PDF do artigo encontrado:", PDF_PATH.exists(), PDF_PATH)
print("Numero de variaveis de entrada consideradas na v10:", len(FULL_FEATURE_COLUMNS))
print("Variaveis de estado:", len(STATE_SENSOR_COLUMNS))
print("Variaveis continuas:", len(CONTINUOUS_SENSOR_COLUMNS))
print("Codigos de class por observacao:", OBSERVATION_CLASS_CODES)
print("Codigos de state por observacao:", OBSERVATION_STATE_CODES)
print("Mapeamento de origem da amostra:", SOURCE_TYPE_MAPPING)

display(attribute_catalog.head(10))
display(event_catalog)
display(manifest.head())

PDF do artigo encontrado: True /home/tiagoriosrocha/Desktop/lstm-w3/2507.01048v1.pdf
Numero de variaveis de entrada consideradas na v10: 27
Variaveis de estado: 9
Variaveis continuas: 18
Codigos de class por observacao: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 101, 102, 103, 105, 106, 107, 108, 109]
Codigos de state por observacao: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Mapeamento de origem da amostra: {'well': 0, 'simulated': 1, 'drawn': 2}


,atributo,papel_no_pipeline,descricao_oficial
0,timestamp,metadado,Instant at which observation was generated
1,ABER-CKGL,metadado,Opening of the GLCK (gas lift choke) [%]
2,ABER-CKP,metadado,Opening of the PCK (production choke) [%]
3,ESTADO-DHSV,estado_discreto,"State of the DHSV (downhole safety valve) [0, ..."
4,ESTADO-M1,estado_discreto,"State of the PMV (production master valve) [0,..."
5,ESTADO-M2,estado_discreto,"State of the AMV (annulus master valve) [0, 0...."
6,ESTADO-PXO,estado_discreto,"State of the PXO (pig-crossover) valve [0, 0.5..."
7,ESTADO-SDV-GL,estado_discreto,"State of the gas lift SDV (shutdown valve) [0,..."
8,ESTADO-SDV-P,estado_discreto,State of the production SDV (shutdown valve) [...
9,ESTADO-W1,estado_discreto,"State of the PWV (production wing valve) [0, 0..."


,class_label,event_name,description,transient_event
0,0,NORMAL,Normal Operation,False
1,1,ABRUPT_INCREASE_OF_BSW,Abrupt Increase of BSW,True
2,2,SPURIOUS_CLOSURE_OF_DHSV,Spurious Closure of DHSV,True
3,3,SEVERE_SLUGGING,Severe Slugging,False
4,4,FLOW_INSTABILITY,Flow Instability,False
5,5,RAPID_PRODUCTIVITY_LOSS,Rapid Productivity Loss,True
6,6,QUICK_RESTRICTION_IN_PCK,Quick Restriction in PCK,True
7,7,SCALING_IN_PCK,Scaling in PCK,True
8,8,HYDRATE_IN_PRODUCTION_LINE,Hydrate in Production Line,True
9,9,HYDRATE_IN_SERVICE_LINE,Hydrate in Service Line,True


,class_label,well_name,start_token,series_id,source_type,file_path,class_label_int
0,0,WELL-00001,20170201010207,0__WELL-00001_20170201010207,well,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0
1,0,WELL-00001,20170201060114,0__WELL-00001_20170201060114,well,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0
2,0,WELL-00001,20170201110124,0__WELL-00001_20170201110124,well,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0
3,0,WELL-00001,20170201160311,0__WELL-00001_20170201160311,well,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0
4,0,WELL-00001,20170201210228,0__WELL-00001_20170201210228,well,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,0


In [3]:
resumo = pd.DataFrame(
    {
        "n_series": [len(manifest)],
        "n_classes_globais": [manifest["class_label"].nunique()],
        "n_fontes": [manifest["source_type"].nunique()],
        "comprimento_medio_da_instancia": [manifest["n_observations"].mean()],
        "comprimento_mediano_da_instancia": [manifest["n_observations"].median()],
    }
)

display(resumo)
display(manifest["class_label"].value_counts().sort_index().rename_axis("classe").to_frame("n_series"))
display(manifest["source_type"].value_counts().rename_axis("fonte").to_frame("n_series"))

KeyError: 'n_observations'

## Leitura academica

A partir desta visao geral, a `versao10` assume que o desafio do `3W` e simultaneamente:

- um problema de classificacao da instancia;
- um problema de acompanhamento temporal da progressao do evento;
- um problema de robustez a artefatos reais do processo industrial.

Isso justifica a passagem para uma arquitetura multitarefa e sensivel ao contexto operacional.